# Week 8: HealthConnect Clinic - Final Model Selection, Documentation & Presentation

## Data Science Track

This notebook finalizes the candidate model developed and tested 
across Weeks 5-7, documenting the final modelling outcome for the 
HealthConnect project's final integration and presentation stage.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(r"C:\Users\HP\Downloads\HealthConnect_Appointment_Data.csv")

df_model = df[df['appointment_outcome'] != 'Cancelled'].copy()
df_model['target_no_show'] = (df_model['appointment_outcome'] == 'No-Show').astype(int)

df_model = df_model.drop(columns=[
    'appointment_id', 'booking_date', 'appointment_date',
    'waiting_time_minutes', 'appointment_outcome'
])

df_model['reminder_channel'] = df_model['reminder_channel'].fillna('None')
median_distance = df_model['distance_to_clinic_km'].median()
df_model['distance_to_clinic_km'] = df_model['distance_to_clinic_km'].fillna(median_distance)

print("df_model shape:", df_model.shape)

df_model shape: (4737, 14)


In [2]:
df_model['prior_no_show_rate'] = np.where(
    df_model['previous_appointments'] > 0,
    df_model['previous_no_shows'] / df_model['previous_appointments'],
    0
)
df_model['has_prior_history'] = (df_model['previous_appointments'] > 0).astype(int)

def categorize_lead_time_v2(days):
    if days <= 7:
        return 'A_0-7'
    elif days <= 14:
        return 'B_8-14'
    elif days <= 30:
        return 'C_15-30'
    else:
        return 'D_31plus'

df_model['booking_lead_category_v2'] = df_model['booking_lead_days'].apply(categorize_lead_time_v2)

print("Feature engineering complete. Shape:", df_model.shape)

Feature engineering complete. Shape: (4737, 17)


In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              roc_auc_score)

categorical_cols_v2 = ['gender', 'age_group', 'appointment_type', 'appointment_day',
                        'appointment_time', 'reminder_sent', 'reminder_channel',
                        'booking_lead_category_v2']

df_encoded = pd.get_dummies(df_model, columns=categorical_cols_v2, drop_first=True)

X = df_encoded.drop(columns=['patient_id', 'target_no_show'])
y = df_encoded['target_no_show']
groups = df_encoded['patient_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

numerical_cols = ['age', 'booking_lead_days', 'previous_appointments',
                   'previous_no_shows', 'distance_to_clinic_km', 'prior_no_show_rate']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba):.3f}")

Accuracy:  0.631
Precision: 0.624
Recall:    0.660
F1-score:  0.642
ROC-AUC:   0.677


In [4]:
results_full = X_test.copy()
results_full['actual'] = y_test.values
results_full['predicted'] = y_pred

def get_category(row):
    if row.get('booking_lead_category_v2_B_8-14', 0) == 1:
        return 'B_8-14'
    elif row.get('booking_lead_category_v2_C_15-30', 0) == 1:
        return 'C_15-30'
    elif row.get('booking_lead_category_v2_D_31plus', 0) == 1:
        return 'D_31plus'
    else:
        return 'A_0-7'

results_full['lead_category'] = results_full.apply(get_category, axis=1)

segment_recall = results_full.groupby('lead_category').apply(
    lambda x: recall_score(x['actual'], x['predicted']) if x['actual'].sum() > 0 else None
)
print("Recall by booking lead time segment:")
print(segment_recall)

Recall by booking lead time segment:
lead_category
A_0-7       0.060606
B_8-14      0.161290
C_15-30     0.336134
D_31plus    0.906667
dtype: float64


## Part 1: Final Integration Readiness

| Requirement | Response |
|---|---|
| Final component | Logistic Regression model predicting patient no-shows, using the refined `booking_lead_category_v2` feature |
| Problem it addresses | Predicting which scheduled appointments are at risk of no-show, to support proactive clinic intervention |
| Current status | Tested and validated through Week 7; suitable for a segmented use case (long-lead-time appointments), not yet general-purpose |
| Improved during Week 7 | Conducted segment-level testing that revealed a severe recall imbalance hidden by aggregate metrics; tested and ruled out an interaction-feature fix with evidence |
| Depends on | Data Analytics (validated findings informing feature decisions) |
| Depended on by | ML Engineering (requires the finalized model interface for pipeline integration) |
| Output to provide | Final model, its documented strengths/limitations, and a clear statement of what it can and cannot be used for |
| Output needed from another track | Confirmation from ML Engineering on pipeline compatibility with the finalized feature set |
| Evidence of readiness | Week 5 baseline, Week 6 model comparison, Week 7 segment-level error analysis and cross-track validation test, all independently reproducible in this notebook |
| Remaining limitations | Severe recall imbalance by booking lead time segment; no hyperparameter tuning or cross-validation performed; synthetic data |

## Tasks 1, 2 & 3: Week 7 Review, Final Model Confirmation, and Baseline Comparison

**Week 7 testing summary:** Systematic testing went beyond aggregate 
metrics into segment-level analysis, revealing that the model's 
overall 66% recall is concentrated almost entirely in long-lead-time 
appointments (91% recall) while short-lead-time appointments are 
barely predicted at all (6-16% recall). An interaction feature 
combining booking lead time and reminder status, informed by a 
validated Data Analytics finding, was tested as a fix and found not 
to resolve the imbalance.

**Final candidate model confirmed:** Logistic Regression, using 
`booking_lead_category_v2` (4-bin, cross-track validated) and 
`reminder_sent` as separate features (the interaction version was 
tested and rejected in Week 7, since it offered no improvement).

**Final performance vs Week 5 baseline:**

| Model | Accuracy | Precision | Recall | F1-score | ROC-AUC |
|---|---|---|---|---|---|
| Week 5 Baseline | 0.631 | 0.624 | 0.660 | 0.642 | 0.677 |
| Final Candidate (Week 8) | 0.631 | 0.624 | 0.660 | 0.642 | 0.677 |

At the aggregate level, the final model is numerically identical to 
the Week 5 baseline. As established across Weeks 6-7, this is an 
honest finding, not a shortfall in effort: multiple refinements were 
tested (a Random Forest alternative in Week 6, a validated interaction 
feature in Week 7), and none improved on the baseline's aggregate 
performance. What did change through this process was the depth of 
understanding of the model's actual behaviour, particularly the 
segment-level imbalance that aggregate metrics alone would have 
concealed entirely.

## Tasks 4 & 5: Final Evaluation and Model Strengths/Weaknesses

**Key evaluation metrics (test set):**
- Accuracy: 0.631
- Precision: 0.624
- Recall: 0.660
- F1-score: 0.642
- ROC-AUC: 0.677

**Segment-level recall by booking lead time:**

| Segment | Recall |
|---|---|
| 0-7 days | 0.061 |
| 8-14 days | 0.161 |
| 15-30 days | 0.336 |
| 31+ days | 0.907 |

**Strengths:**
- Strong, reliable performance for long-lead-time appointments (91% 
  recall), where the model provides genuine early-warning value.
- No evidence of overfitting (train/test performance nearly 
  identical), so this performance should generalise reasonably to 
  similar future data.
- Fully interpretable: every feature's contribution can be explained 
  directly to non-technical stakeholders.
- Built on a cross-track-validated feature set, not just internal 
  assumptions.

**Weaknesses:**
- Severe recall imbalance: the model is nearly blind to no-shows in 
  short-lead-time appointments (6-16% recall), despite those 
  appointments still containing meaningful no-show rates (29.5% for 
  0-7 days).
- A validated attempt to fix this (interaction feature) did not work, 
  suggesting the issue is structural (the linear model's decision 
  boundary), not simply a matter of better features.
- No hyperparameter tuning or cross-validation has been performed at 
  any stage.

## Tasks 6 & 7: Modelling Decisions and Business Suitability

**Key modelling decisions across the project:**
1. Excluded Cancelled appointments from the target (Week 4-5), since a 
   cancellation is a distinct, proactive behaviour from a silent 
   no-show.
2. Selected Logistic Regression over Random Forest (Week 6) because it 
   matched the baseline's recall while remaining interpretable, even 
   though Random Forest showed marginally better accuracy.
3. Refined `booking_lead_category` from 3 to 4 bins (Week 6) using a 
   cross-track-validated finding.
4. Tested and rejected an interaction feature (Week 7) after evidence 
   showed it didn't address the segment imbalance.

**Business suitability:** The model is suitable for a **segmented use 
case**: flagging no-show risk for appointments booked well in advance 
(31+ days), where it performs strongly. It is **not yet suitable** as 
a general-purpose no-show predictor across all booking timeframes, 
since it currently provides almost no predictive value for short-notice 
appointments, arguably the harder and more operationally costly case 
for HealthConnect to manage.

## Task 8: Incorporating Data Analytics Findings

Two validated findings from the Data Analytics track informed this 
project's final model:

1. **Week 6:** A 4-bin breakdown of no-show rate by booking lead time 
   (0-7, 8-14, 15-30, 31+ days), used to refine `booking_lead_category` 
   from its original 3-bin version. Independently reproduced and 
   confirmed to match exactly.
2. **Week 7:** A combined breakdown showing the lead-time pattern holds 
   even when reminder status is considered, with reminders associated 
   with lower no-show rates across every lead-time group. This was 
   tested as an interaction feature; the test showed no improvement, a 
   validated negative result that nonetheless confirmed the underlying 
   pattern was real, just not usable in that particular encoded form 
   for this model.

Both findings shaped the final feature set and the understanding of 
the model's limitations, even though the second did not lead to a 
feature change.

## Task 9: Communication to ML Engineering

The following final model requirements were prepared for the ML 
Engineering track:

- **Final model:** Logistic Regression, 31 features post-encoding, 
  requiring `StandardScaler`-transformed numerical inputs (`age`, 
  `booking_lead_days`, `previous_appointments`, `previous_no_shows`, 
  `distance_to_clinic_km`, `prior_no_show_rate`) and one-hot encoded 
  categorical inputs.
- **Target encoding:** 1 = No-Show, 0 = Attended. Stable since Week 6, 
  no changes.
- **Known limitation to flag:** Severe recall imbalance by booking 
  lead time segment; any deployment should account for this rather 
  than treating the model's output as uniformly reliable across all 
  appointment types.

**Note on this handoff:** This requirement summary was prepared and 
made available for the ML Engineering track throughout Weeks 6-8. In 
practice, the direct exchange planned earlier in the project did not 
materialise into an actioned collaboration, unlike the sustained, 
validated collaboration with the Data Analytics track. This is 
recorded honestly here rather than overstated: the requirements were 
ready and available, but a completed two-way integration with ML 
Engineering did not occur within this project's timeline.

## Task 10: Final Model Limitations and Risks

- **Segment imbalance (primary limitation):** Recall ranges from 6% 
  (0-7 days) to 91% (31+ days) across booking lead time segments.
- **Validated negative result:** An interaction feature, informed by a 
  cross-track finding, did not resolve this imbalance, suggesting the 
  limitation is structural to the linear model, not merely a feature 
  gap.
- **No hyperparameter tuning or cross-validation performed** at any 
  stage of the project.
- **Single train/test split:** all performance figures rely on one 
  patient-aware split.
- **Synthetic data:** real-world generalisation remains unverified.
- **Incomplete cross-track loop with ML Engineering:** the model's 
  requirements were documented and shared but not confirmed through an 
  actioned two-way exchange.

## Task 11: What This Model Can and Cannot Be Used For

**Can be used for:** Flagging no-show risk for appointments booked 31 
or more days in advance, where the model correctly identifies 91% of 
genuine no-shows. This is directly actionable for HealthConnect, e.g. 
prioritising reminder outreach or overbooking strategy for this 
segment.

**Cannot currently be used for:** Reliably predicting no-shows for 
appointments booked within 30 days, and especially not within 7 days, 
where the model catches as few as 6% of genuine no-shows. Deploying 
this model as a general-purpose no-show predictor without this caveat 
would give HealthConnect false confidence in short-notice risk 
coverage.

## Task 12: Model Section for Final Presentation

The following talking points summarise the model story for the 
individual video presentation:

1. Built a baseline (Week 5) predicting patient no-shows, achieving 
   66% recall.
2. Tested improvements against that baseline (Week 6): a cross-track 
   validated feature refinement and a Random Forest alternative, 
   neither improved aggregate performance, an honest early signal.
3. Discovered through segment-level testing (Week 7) that the 
   aggregate 66% recall was misleading, the model performs strongly 
   for long-lead-time appointments (91% recall) but very poorly for 
   short-notice ones (6% recall).
4. Tested a second cross-track validated hypothesis (interaction 
   feature) to fix this, ruled it out with evidence.
5. Finalised the model with a clear, honest statement of what it can 
   and cannot do for HealthConnect.

## Task 13: Model Summary for Non-Technical Stakeholders

HealthConnect Clinic wanted to know which patients are likely to miss 
their appointments. We built a model that looks at things like how far 
in advance an appointment was booked, whether a reminder was sent, and 
a patient's past attendance history, to estimate the risk of a 
no-show.

The model works well for appointments booked far in advance: it 
correctly flags about 9 out of 10 genuine no-shows in that group. This 
means HealthConnect could use it right now to prioritise reminder 
calls or texts for patients with appointments booked more than a month 
out.

The model does not yet work well for appointments booked at short 
notice, catching only about 1 in 15 genuine no-shows in that group. We 
tested a specific fix for this using a validated finding from a 
teammate on the Data Analytics side, but it did not solve the problem, 
a useful finding in itself, since it tells us the fix needs to happen 
at the model level, not the data level.

**Bottom line:** Ready to help with long-notice appointments today. 
Short-notice appointments need more work before this model should be 
trusted for them.

## Task 14: Final Notebook Organization

This notebook is organised to walk a reader through the complete final 
modelling story: Part 1 (integration readiness) → Tasks 1-3 (Week 7 
review, final model, baseline comparison) → Tasks 4-5 (metrics, 
strengths/weaknesses) → Tasks 6-7 (decisions, business suitability) → 
Task 8 (Data Analytics findings) → Task 9 (ML Engineering 
communication) → Tasks 10-11 (limitations, usage boundaries) → Tasks 
12-13 (presentation materials) → this section.

All code cells reproduce results already validated across Weeks 5-7, 
and every claim in this notebook is backed by a number calculated 
directly from the dataset, no results are asserted without 
accompanying code and output.